# Laboratorio 4 — Electrocardiograma (ECG)
## Reposo, Hiperventilación, Hipoventilación y Actividad Aeróbica — Derivaciones DI, DII y DIII

**Curso:** Introducción a Señales Biomédicas
**Autora:** Astrid Mejia

Los archivos `.txt` (formato OpenSignals) deben estar en la carpeta `Datos/`, junto a este notebook.


In [ ]:
pip install opensignalsreader

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, iirnotch, find_peaks
from opensignalsreader import OpenSignalsReader

# ── Función reutilizable de análisis ─────────────────────────────────────────
def bandpass(sig, low, high, fs, order=4):
    b, a = butter(order, [low / (0.5 * fs), high / (0.5 * fs)], btype='band')
    return filtfilt(b, a, sig)

def analizar_ecg(path, titulo):
    # ── Lectura ───────────────────────────────────────────────────────────
    acq = OpenSignalsReader(path)
    fs = acq.sampling_rate

    ecg = acq.signal()['ECG'].astype(float)
    ecg -= ecg.mean()
    t = np.arange(len(ecg)) / fs

    # ── Filtros: pasa-banda 0.5-40 Hz + notch 60 Hz (red eléctrica) ─────────
    ecg = bandpass(ecg, 0.5, 40, fs)
    b_n, a_n = iirnotch(60, 30, fs)
    ecg = filtfilt(b_n, a_n, ecg)

    # ── Detección de picos R y frecuencia cardíaca ──────────────────────────
    peaks, _ = find_peaks(ecg, distance=int(0.4 * fs), height=0.4 * ecg.max())
    rr_ms = np.diff(peaks) / fs * 1000
    fc_media = (60 / (rr_ms / 1000)).mean()

    print(f'{titulo} -> Frecuencia cardíaca media: {fc_media:.1f} bpm  (duración: {len(ecg)/fs:.1f} s)')

    # ── Gráficos: señal completa + zoom 5 s ─────────────────────────────────
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))
    fig.suptitle(f'Señal ECG – {titulo}', fontsize=13, fontweight='bold')

    ax1.plot(t, ecg, color='#1a6bb5', lw=0.7)
    ax1.plot(t[peaks], ecg[peaks], 'rx', ms=6, label='Picos R')
    ax1.set_title('Señal completa', fontsize=11)
    ax1.set_xlabel('Tiempo [s]'); ax1.set_ylabel('Amplitud [mV]')
    ax1.legend(fontsize=9)
    ax1.grid(True, linestyle='--', alpha=0.5)

    samples = 5 * fs
    ax2.plot(t[:samples], ecg[:samples], color='#1a6bb5', lw=0.9)
    ax2.plot(t[peaks[peaks < samples]], ecg[peaks[peaks < samples]], 'rx', ms=6)
    ax2.set_title('Zoom – primeros 5 segundos', fontsize=11)
    ax2.set_xlabel('Tiempo [s]'); ax2.set_ylabel('Amplitud [mV]')
    ax2.grid(True, linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()

    return {'fc_media': fc_media, 'duracion_s': len(ecg) / fs}

resultados = {}

## 1. Derivación DI

### 1.1 Reposo

In [ ]:
resultados[('DI', 'Reposo')] = analizar_ecg('Datos/DI_BASAL.txt', 'DI – Reposo')

### 1.2 Hiperventilación

In [ ]:
resultados[('DI', 'Hiperventilación')] = analizar_ecg('Datos/DI_HIPER.txt', 'DI – Hiperventilación')

### 1.3 Hipoventilación

In [ ]:
resultados[('DI', 'Hipoventilación')] = analizar_ecg('Datos/DI_HIPO.txt', 'DI – Hipoventilación')

### 1.4 Actividad aeróbica

In [ ]:
resultados[('DI', 'Actividad aeróbica')] = analizar_ecg('Datos/DI_ACTI.txt', 'DI – Actividad aeróbica')

## 2. Derivación DII

### 2.1 Reposo

In [ ]:
resultados[('DII', 'Reposo')] = analizar_ecg('Datos/DII_BASAL.txt', 'DII – Reposo')

### 2.2 Hiperventilación

In [ ]:
resultados[('DII', 'Hiperventilación')] = analizar_ecg('Datos/DII_HIPER.txt', 'DII – Hiperventilación')

### 2.3 Hipoventilación

In [ ]:
resultados[('DII', 'Hipoventilación')] = analizar_ecg('Datos/DII_HIPO.txt', 'DII – Hipoventilación')

### 2.4 Actividad aeróbica

In [ ]:
resultados[('DII', 'Actividad aeróbica')] = analizar_ecg('Datos/DII_ACTI.txt', 'DII – Actividad aeróbica')

## 3. Derivación DIII

### 3.1 Reposo

In [ ]:
resultados[('DIII', 'Reposo')] = analizar_ecg('Datos/DIII_BASAL.txt', 'DIII – Reposo')

### 3.2 Hiperventilación

In [ ]:
resultados[('DIII', 'Hiperventilación')] = analizar_ecg('Datos/DIII_HIPER.txt', 'DIII – Hiperventilación')

### 3.3 Hipoventilación

In [ ]:
resultados[('DIII', 'Hipoventilación')] = analizar_ecg('Datos/DIII_HIPO.txt', 'DIII – Hipoventilación')

### 3.4 Actividad aeróbica

In [ ]:
resultados[('DIII', 'Actividad aeróbica')] = analizar_ecg('Datos/DIII_ACTI.txt', 'DIII – Actividad aeróbica')

## 4. Resumen de frecuencia cardíaca

Frecuencia cardíaca media (bpm) obtenida en cada combinación derivación–estadio.

In [ ]:
for (der, estado), r in resultados.items():
    print(f'{der:5s} {estado:20s} FC media: {r["fc_media"]:.1f} bpm  (duración: {r["duracion_s"]:.1f} s)')